# 04 — Circuit Breakers: Automatic Failure Isolation

## What is a Circuit Breaker?

A **circuit breaker** prevents a failing component from cascading failures across the system. It models the state of an agent node as a switch with three states:

```
             consecutive failures ≥ trip_threshold
  CLOSED ─────────────────────────────────────────► OPEN
  (normal)                                          (tripped)
     ▲                                                  │
     │        after recovery_executions successes       │
  HALF_OPEN ◄─────────────────────────────────────────────
  (testing probe)
```

| State | Behaviour |
|-------|-----------|
| **CLOSED** | Requests pass through normally |
| **OPEN** | Requests are immediately redirected to `fallback_agent` |
| **HALF_OPEN** | A single probe request is allowed; success → CLOSED, failure → OPEN |

## How it Works in Multigen

Each node can declare:
- `fallback_agent` — the agent to use when the CB is OPEN
- The graph-level `circuit_breaker` config sets `trip_threshold` (failures to trip) and `recovery_executions` (successes to reset)

## Notebook Goal

Build a graph where `GuardrailAgent` has a fallback of `EchoAgent` and a trip threshold of 2. We will:
1. Show the graph configuration
2. Run the workflow
3. Query health to see CB status
4. Verify fallback ran when CB opened

In [ ]:
import time
import json
import pprint

from multigen import SyncMultigenClient
from multigen.dsl import GraphBuilder

ORCHESTRATOR_URL = "http://localhost:8009"
client = SyncMultigenClient(base_url=ORCHESTRATOR_URL, timeout=60.0)
assert client.ping(), "Orchestrator not reachable."
print("Connected.")

## 1. Build the Graph with Circuit Breaker Configuration

Key settings:
- `GuardrailAgent` node has `.fallback("EchoAgent")` — EchoAgent will run when the CB is OPEN
- `.circuit_breaker(trip_threshold=2, recovery_executions=3)` — trip after 2 consecutive failures, recover after 3 successes

In [ ]:
graph_def = (
    GraphBuilder()
    # Primary processing node — runs normally
    .node("screen")
        .agent("SkillMatcherAgent")
        .params(job_title="Backend Engineer")
        .timeout(30)
        .done()
    # GuardrailAgent node with EchoAgent as fallback
    # If GuardrailAgent fails trip_threshold times, EchoAgent takes over
    .node("guardrail")
        .agent("GuardrailAgent")
        .params(policy="compliance_check", strict=True)
        .fallback("EchoAgent")   # <-- fallback agent when CB is OPEN
        .timeout(15)
        .done()
    # Final reporting node
    .node("report")
        .agent("ReportCompilerAgent")
        .params(format="json")
        .timeout(20)
        .done()
    # Edges
    .edge("screen",   "guardrail")
    .edge("guardrail", "report")
    # Graph-level circuit breaker: trip after 2 failures, recover after 3 successes
    .entry("screen")
    .max_cycles(10)
    .circuit_breaker(trip_threshold=2, recovery_executions=3)
    .build()
)

print("Circuit breaker config:")
print(json.dumps(graph_def["circuit_breaker"], indent=2))
print("\nNodes:")
for n in graph_def["nodes"]:
    fa = n.get("fallback_agent", "(none)")
    print(f"  {n['id']:20s} agent={n.get('agent','?'):25s} fallback={fa}")

## 2. Circuit Breaker State Machine — Detailed Explanation

```
Initial state: CLOSED
                    │
                    ▼
          ┌─────────────────┐
          │     CLOSED      │  ← All calls go to GuardrailAgent
          │  (normal path)  │
          └─────────────────┘
               │ failure count >= trip_threshold (2)
               ▼
          ┌─────────────────┐
          │      OPEN       │  ← All calls redirected to EchoAgent (fallback)
          │  (tripped CB)   │     No calls reach GuardrailAgent
          └─────────────────┘
               │ after recovery_executions (3) probe successes
               ▼
          ┌─────────────────┐
          │   HALF_OPEN     │  ← One probe call to GuardrailAgent
          │  (testing)      │     Success → CLOSED | Failure → OPEN
          └─────────────────┘
```

The CB is global to the graph (not per-node). `cb_trips_total` in health tracks cumulative trips.

## 3. Run the Workflow

In [ ]:
payload = {
    "resume_text": "Bob Lee, 2 years DevOps, Terraform, AWS, Python.",
    "candidate_id": "candidate-003",
}

response = client.run_graph(graph_def=graph_def, payload=payload)
instance_id = response.instance_id
print(f"Launched — instance_id: {instance_id}")

## 4. Poll Health During Execution

We poll `get_health()` every 2 seconds to observe the circuit-breaker status evolving in real-time.

In [ ]:
print("Polling health every 2 seconds for 20 seconds...\n")
print(f"{'Time':>6s}  {'interrupted':>12s}  {'pending':>8s}  {'cb_trips':>9s}  {'errors'}")
print("-" * 60)

start = time.time()
for _ in range(10):
    try:
        h = client.get_health(instance_id)
        elapsed = time.time() - start
        print(
            f"{elapsed:6.1f}s  "
            f"{str(h.interrupted):>12s}  "
            f"{h.pending_count:>8d}  "
            f"{h.cb_trips_total:>9d}  "
            f"{h.errors}"
        )
    except Exception as e:
        print(f"  (health query failed: {e})")
    time.sleep(2)

## 5. Inspect Final State — Did the Fallback Run?

When the circuit breaker trips:
- The `guardrail` node's output will contain metadata indicating the fallback agent executed
- `metrics.circuit_breaker_trips` will be > 0

In [ ]:
# Allow time for completion
time.sleep(3)

final_state = client.get_state(instance_id)
metrics = client.get_metrics(instance_id)

print("=" * 55)
print("Final Node Outputs")
print("=" * 55)
for ns in final_state.nodes:
    print(f"\nNode: {ns.node_id}")
    print("-" * 40)
    # Check if fallback was used
    if ns.output.get("fallback_used") or ns.output.get("agent_used") == "EchoAgent":
        print("  *** FALLBACK AGENT (EchoAgent) WAS USED ***")
    pprint.pprint(ns.output, indent=4)

print("\n" + "=" * 55)
print("Metrics")
print("=" * 55)
print(f"  nodes_executed       : {metrics.nodes_executed}")
print(f"  circuit_breaker_trips: {metrics.circuit_breaker_trips}")
print(f"  error_count          : {metrics.error_count}")
print(f"  dead_letter_count    : {metrics.dead_letter_count}")

## 6. Summary — Circuit Breaker Configuration Cheat Sheet

```python
# Per-node: declare a fallback agent
.node("my_node")
    .agent("PrimaryAgent")
    .fallback("FallbackAgent")   # used when CB is OPEN
    .done()

# Graph-level: configure CB parameters
.circuit_breaker(
    trip_threshold=2,        # failures before CB opens
    recovery_executions=3,   # successes before CB closes
)
```

**Health fields to monitor:**
- `health.cb_trips_total` — total number of times CB has tripped
- `health.errors` — list of recent error messages
- `health.dead_letters` — nodes that exhausted all retries and fallbacks

**Next**: See `05_reflection_loops.ipynb` for confidence-based auto-reflection.

In [ ]:
client.close()
print("Client closed.")